# Lab F2. Comparing forecasts

Two forecasts, or twenty: which is better, and is the difference more than luck? This lab compares a handful of
simple models of a series that is only a little predictable, which is where the choice of test matters.

**Help:** for any pymetrix function, type its name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call. The last cell lists every pymetrix function, class and method this lab uses.

Topics: the Diebold-Mariano test; nested models and the Clark-West test; conditional predictive ability
(Giacomini-White); forecast encompassing; many forecasts at once: SPA (Reality Check), the model confidence set
and StepM (Elliott and Timmermann, chapter 17).

The data: 240 quarters of $y_t = 0.5 + 0.3\,y_{t-1} + \varepsilon_t$, $\varepsilon_t \sim N(0, 1)$.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from pymetrix import Backtest, Dataset, Equation, ExponentialSmoothing, backtest
from pymetrix.stats import clark_west

rng = np.random.default_rng(17)
n = 240
y = np.zeros(n)
y[0] = 0.5 / 0.7
for t in range(1, n):
    y[t] = 0.5 + 0.3 * y[t - 1] + rng.standard_normal()
data = Dataset.from_frame(pd.DataFrame({"y": y}, index=pd.period_range("1965Q1", periods=n, freq="Q")), source="simulated AR(1)")

## 1. Five forecasts

The historical mean (a model with no dynamics, nested in the AR(1)), the AR(1), the AR(2) and AR(4) (which nest
the AR(1)), and simple exponential smoothing. Each is re-estimated at every origin from 1995Q1 to 2023Q4 and
forecasts one to four quarters ahead.

In [ ]:
origins = ("1995Q1", "2023Q4")
fits = {
    "mean": Equation("y ~ 1").fit(data),
    "AR(1)": Equation("y ~ y[-1]").fit(data),
    "AR(2)": Equation("y ~ y[-1] + y[-2]").fit(data),
    "AR(4)": Equation("y ~ y[-1] + y[-2] + y[-3] + y[-4]").fit(data),
    "smoothing": ExponentialSmoothing("y").fit(data),
}
bts = {name: backtest(fit, data, origins, horizon=4, label=name) for name, fit in fits.items()}
pd.DataFrame({name: bt.scores(["y"])["RMSE"].to_numpy() for name, bt in bts.items()}, index=pd.Index(range(1, 5), name="horizon"))

## 2. Two forecasts: Diebold-Mariano

The AR(1) is the true model, but it forecasts only a little better than the mean: most of the variance of $y$ is
unpredictable noise. Is the AR(1)'s advantage one quarter ahead significant?

In [ ]:
dm = bts["AR(1)"].compare(bts["mean"], "y", 1)
dm

## 3. Nested models: Clark-West

The mean is the AR(1) with $\phi = 0$. Under the null that $\phi = 0$, the AR(1) still estimates $\phi$, which
adds noise to its forecasts, so its MSE is expected to be *higher* than the mean's: the Diebold-Mariano test,
centred on equal MSE, is undersized and lacks power. Clark and West adjust the larger model's MSE for that noise,
by $(\hat y_{\text{small}} - \hat y_{\text{large}})^2$, and test one-sided.

In [ ]:
cw = bts["mean"].compare(bts["AR(1)"], "y", 1, test="clark-west")
cw

**Check it yourself.** The Clark-West statistic is a t-statistic on
$f_t = e_{\text{small},t}^2 - \left[e_{\text{large},t}^2 - (\hat y_{\text{small},t} - \hat y_{\text{large},t})^2\right]$.

In [ ]:
small = bts["mean"].errors.query("horizon == 1").set_index("target")
large = bts["AR(1)"].errors.query("horizon == 1").set_index("target")
f = small["error"] ** 2 - (large["error"] ** 2 - (small["forecast"] - large["forecast"]) ** 2)
t_by_hand = f.mean() / np.sqrt(((f - f.mean()) ** 2).mean() / len(f))
assert np.isclose(t_by_hand, cw.statistic) and np.isclose(stats.norm.sf(t_by_hand), cw.p_value)
print(f"Clark-West by hand: {t_by_hand:.4f}; pymetrix: {cw.statistic:.4f}")

### How often does each test reject when it should not?

A small Monte Carlo: simulate series with *no* predictability ($\phi = 0$), forecast them with the mean and the
AR(1) from an expanding window, and count how often each test rejects at 10%. A test with the right size rejects
about 10% of the time. (A few seconds.)

In [ ]:
def recursive_forecasts(series, first):
    """One-step forecasts from the mean and from an AR(1), each re-estimated on the data to each origin."""
    small, large, actual = [], [], []
    for t in range(first, len(series) - 1):
        past = series[: t + 1]
        X = np.column_stack([np.ones(t), past[:-1]])
        b = np.linalg.lstsq(X, past[1:], rcond=None)[0]
        small.append(past.mean())
        large.append(b[0] + b[1] * past[-1])
        actual.append(series[t + 1])
    return np.array(actual), np.array(small), np.array(large)


mc = np.random.default_rng(0)
rejections = {"Diebold-Mariano (one-sided)": 0, "Clark-West": 0}
reps = 300
for _ in range(reps):
    a, s, l = recursive_forecasts(mc.standard_normal(160), 60)
    d = (a - l) ** 2 - (a - s) ** 2  # negative when the AR(1) wins
    dm_t = d.mean() / np.sqrt(d.var() / len(d))
    rejections["Diebold-Mariano (one-sided)"] += int(stats.norm.cdf(dm_t) < 0.10)
    rejections["Clark-West"] += int(clark_west(a, s, l).p_value < 0.10)
size = {k: v / reps for k, v in rejections.items()}
size

**Question.** Which test is closer to 10%? What does that mean for the Diebold-Mariano test's power against a
small amount of predictability like the AR(1) here?

## 4. Conditional predictive ability and encompassing

Giacomini and White ask whether one forecast is better *given what is known at the origin*, here the last loss
differential; their test also covers forecasts from rolling windows and nested models, because it is about the
forecasting method, estimation included. Encompassing asks whether a forecast already contains all the useful
information in another: if not, a combination would beat it.

In [ ]:
gw = bts["AR(1)"].compare(bts["smoothing"], "y", 1, test="giacomini-white")
enc_ar = bts["AR(1)"].encompasses(bts["smoothing"], "y", 1)
enc_es = bts["smoothing"].encompasses(bts["AR(1)"], "y", 1)
pd.DataFrame({"statistic": [gw.statistic, enc_ar.statistic, enc_es.statistic], "p-value": [gw.p_value, enc_ar.p_value, enc_es.p_value]},
             index=["Giacomini-White: AR(1) vs smoothing", "AR(1) encompasses smoothing", "smoothing encompasses AR(1)"])

## 5. Many forecasts at once

Testing the best of five models against a benchmark one at a time overstates the evidence: with enough models,
one will look better by chance. Hansen's SPA test asks whether *any* model beats the benchmark, allowing for the
search; the model confidence set keeps every model that could be the best; StepM says which models beat the
benchmark, controlling the chance of any false claim. All three use a bootstrap that keeps the losses' time
dependence. They need the `arch` package (`pip install pymetrix[arch]`).

In [ ]:
losses = {name: bt.losses("y", 1) for name, bt in bts.items()}
try:
    from pymetrix.stats import model_confidence_set, spa, stepm

    spa_test = spa(losses["mean"], {k: v for k, v in losses.items() if k != "mean"}, reps=500)
    mcs = model_confidence_set(losses, alpha=0.1, reps=500)
    step = stepm(losses["mean"], {k: v for k, v in losses.items() if k != "mean"}, alpha=0.1, reps=500)
    print(spa_test, mcs, step, sep="\n\n")
except ImportError as exc:  # pragma: no cover - arch not installed
    spa_test = mcs = step = None
    print(f"skipped: {exc}")

**Questions.**

1. The AR(4) nests the AR(1). Test whether it forecasts better with Clark-West (`bts["AR(1)"].compare(bts["AR(4)"],
   "y", 1, test="clark-west")`). Then compare their RMSEs. What is the cost of the extra lags?
2. Repeat the comparisons four quarters ahead (horizon 4). Why does the gap between the AR(1) and the mean close?
3. Clark-West found the AR(1) significantly better than the mean, but SPA and StepM find no model better than the
   mean. Are these results inconsistent? What does each test allow for that the other does not?

In the app: **Forecast → Compare backtests** runs Diebold-Mariano, Giacomini-White or Clark-West;
**Forecast → Compare many forecasts** runs SPA, the model confidence set and StepM on backtests chosen from a list.

## pymetrix used in this lab

Every pymetrix function, class and method this lab calls, with its arguments and the first line of its documentation. For the full documentation, type the name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call.

### From `pymetrix`

| | |
|---|---|
| `backtest(fitted, data, origins, horizon, vintages, variable, label, window)` | Re-estimate at each origin from `origins[0]` to `origins[1]` and forecast `horizon` periods ahead. |
| `Equation(text, params, family, priors, name, errors, variance)` | An equation written as text: `"c ~ y + c[-1]"` to estimate, or `"y = c + i + g"` (an identity). |
| `ExponentialSmoothing(series, trend, damped, seasonal, error, backend_options)` | Exponential smoothing (ETS), estimated by maximum likelihood (statsmodels' ETSModel). |
| `model_confidence_set(losses, alpha, horizon, reps, block, seed)` | The model confidence set of Hansen, Lunde and Nason (2011): the models that contain the best one with probability 1 - `alpha`, from their losses (one series per model), with the stationary bootstrap (arch's `MCS`, range statistic). |
| `spa(benchmark, models, horizon, reps, block, seed)` | Hansen's (2005) test for superior predictive ability: does any of `models` have lower expected loss than the `benchmark`? From the losses (one series per model), with the stationary bootstrap (arch's `SPA`). |

### From `pymetrix.stats`

| | |
|---|---|
| `clark_west(actual, small, large, horizon)` | Clark-West (2007) test for nested models: does the larger model forecast better than the smaller one it nests? |
| `stepm(benchmark, models, alpha, horizon, reps, block, seed)` | Romano and Wolf's (2005) stepwise multiple test (StepM): which of `models` have lower expected loss than the `benchmark`, controlling the familywise error rate at `alpha`? From the losses, with the stationary bootstrap (arch's `StepM`). |

### Methods and properties of the results

| | |
|---|---|
| `Backtest.compare(other, variable, horizon, loss, test, tau, a)` | Test whether these forecasts and `other`'s are equally accurate at one horizon, on the targets both forecast. |
| `Backtest.encompasses(other, variable, horizon)` | Do these forecasts encompass `other`'s (Harvey, Leybourne and Newbold), on the targets both forecast? |
| `Backtest.losses(variable, horizon, loss, tau, a)` | The loss of each forecast of `variable` at `horizon`, by target period: the `"squared"` or `"absolute"` error, the asymmetric `"lin-lin"` (with `tau`) or `"linex"` (with `a`) loss of the error, the `"crps"`, or the negative `"log score"` (so that lower is better for all of them). |
| `Backtest.scores(variables)` | n, RMSE, MAE and mean error by variable and horizon (a two-level index). |
| `Dataset.from_frame(df, time, entity, freq, variables, meta, source, start)` | Build a Dataset from a DataFrame, naming its time and entity columns. |
| `Equation.fit(data, span, where, method, weights, missing)` | Estimate the equation on `data`: by least squares unless `method` says otherwise (`OLS(cov="hac")`, `IV(...)`, `ML()`, `Ridge()`, ...). |
| `ExponentialSmoothing.fit(data, span)` | Estimate the smoothing parameters and initial states by maximum likelihood on the periods of `span` (first, last), which must hold an unbroken run of the series. |